# GOLD 01. CREACION TABLAS

## 1. Configuración

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Gold_Creacion") \
    .getOrCreate()

FORMATO   = "PARQUET"                 
RUTA_GOLD = "gs://tfm_unir1/gold"
ESQUEMA   = "gold"

# Funciones auxiliares: devuelven el nombre completo de la tabla y su ubicación
def tabla(nombre):
    return f"{ESQUEMA}.{nombre}" if ESQUEMA else nombre

def ubicacion(nombre):
    return f"LOCATION '{RUTA_GOLD}/{nombre}'" if RUTA_GOLD else ""

26/11/24 20:48:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 2. Creación Esquema GOLD

In [3]:
if ESQUEMA:

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {ESQUEMA} LOCATION '{RUTA_GOLD}'")

spark.sql ("SHOW DATABASES").show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+
|namespace|
+---------+
|  default|
|     gold|
+---------+



## 3. Dimensiones

In [16]:
# dim_tiempo

spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_tiempo')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_tiempo")}(
        id_tiempo INT NOT NULL,
        fecha_hora TIMESTAMP NOT NULL,
        fecha DATE NOT NULL,
        anio INT,
        mes INT,
        dia INT,
        hora INT,
        es_fin_de_semana BOOLEAN,
        es_festivo BOOLEAN,
        nombre_festivo STRING,
        es_evento BOOLEAN,
        nombre_evento STRING,
        es_confinamiento BOOLEAN,
        dia_semana INT,
        nombre_dia STRING,
        nombre_mes STRING,
        trimestre INT
    )
    USING {FORMATO}
    {ubicacion("dim_tiempo")}
    """)

DataFrame[]

In [17]:
# dim_estacion

spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_estacion')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_estacion")}(
        id_estacion STRING NOT NULL,
        codigo_eea STRING,
        nombre_estacion STRING,
        tipo_estacion STRING,
        zona STRING,
        latitud DOUBLE,
        longitud DOUBLE,
        altitud DOUBLE,
        fecha_alta DATE,
        fecha_baja DATE,
        activa BOOLEAN,
        id_distrito INT
    )
    USING {FORMATO}
    {ubicacion("dim_estacion")}
    """)

DataFrame[]

In [18]:
# dim_contaminante
spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_contaminante')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_contaminante")}(
        id_contaminante INT NOT NULL,
        nombre STRING,
        unidad STRING,
        indicador STRING,
        limite_ue_actual DOUBLE,
        objetivo_ue_2030 DOUBLE,
        guia_oms_2021 DOUBLE
    )
    USING {FORMATO}
    {ubicacion("dim_contaminante")}
    """)

DataFrame[]

In [19]:
# dim_espira
spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_espira')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_espira")}(
        id_espira STRING NOT NULL,
        latitud DOUBLE,
        longitud DOUBLE,
        id_distrito INT
    )
    USING {FORMATO}
    {ubicacion("dim_espira")}
    """)

DataFrame[]

In [20]:
# dim_distrito
spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_distrito')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_distrito")}(
        id_distrito INT NOT NULL,
        distrito STRING,
        superficie_km2 DOUBLE,
        latitud DOUBLE,
        longitud DOUBLE
    )
    USING {FORMATO}
    {ubicacion("dim_distrito")}
    """)

DataFrame[]

In [3]:
# dim_sonometro
spark.sql(f"DROP TABLE IF EXISTS {tabla('dim_sonometro')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("dim_sonometro")}(
        id_sonometro STRING NOT NULL,
        red STRING,
        latitud DOUBLE,
        longitud DOUBLE,
        id_distrito INT
    )
    USING {FORMATO}
    {ubicacion("dim_sonometro")}
    """)

26/11/24 20:50:19 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/dim_sonometro was not found. Was it deleted very recently?
26/11/24 20:50:19 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

## 4. Tablas de hechos

In [21]:
# fact_aire
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_aire')}")


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_aire")}(
        id_tiempo INT NOT NULL,
        id_estacion STRING NOT NULL,
        id_contaminante INT NOT NULL,
        id_distrito INT,
        concentracion DOUBLE
    )
    USING {FORMATO}
    {ubicacion("fact_aire")}
    """)

DataFrame[]

In [22]:
# fact_meteo
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_meteo')}")


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_meteo")}(
        id_tiempo INT NOT NULL,
        id_estacion STRING NOT NULL,
        temperatura DOUBLE, 
        humedad_relativa DOUBLE,
        velocidad_viento DOUBLE,
        direccion_viento DOUBLE,
        precipitacion DOUBLE,
        presion DOUBLE,
        radiacion_solar DOUBLE
       
    )
    USING {FORMATO}
    {ubicacion("fact_meteo")}
    """)

DataFrame[]

In [23]:
# fact_trafico
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_trafico')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_trafico")}(
        id_tiempo INT NOT NULL,
        id_espira STRING NOT NULL,
        id_distrito INT NOT NULL,
        intensidad DOUBLE
       
    )
    USING {FORMATO}
    {ubicacion("fact_trafico")}
    """)

DataFrame[]

In [24]:
# fact_prediccion
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_prediccion')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_prediccion")}(
        modelo STRING NOT NULL, -- gradient_boosting, calendario, tendencia
        indicador STRING NOT NULL, -- no2, trafico, pm10, ruido
        horizonte STRING, -- 24h, mes, anio, 2030
        granularidad STRING,
        fecha_ejecucion DATE,
        fecha_prevista DATE NOT NULL,
        id_tiempo INT, -- solo en predicciones horarias
        id_estacion STRING, -- vacio si es toda la ciudad
        valor_previsto DOUBLE,
        valor_bajo DOUBLE,
        valor_alto DOUBLE       
    )
    USING {FORMATO}
    {ubicacion("fact_prediccion")}
    """)

DataFrame[]

In [25]:
# fact_poblacion
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_poblacion')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_poblacion")}(
        id_distrito INT NOT NULL,
        anio INT NOT NULL,
        poblacion INT
    )
    USING {FORMATO}
    {ubicacion("fact_poblacion")}
    """)

DataFrame[]

In [26]:
# fact_zona_verde
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_zona_verde')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_zona_verde")}(
        id_distrito INT NOT NULL,
        m2_verde DOUBLE,
        n_espacios INT
    )
    USING {FORMATO}
    {ubicacion("fact_zona_verde")}
    """)

DataFrame[]

In [1]:
# Cambio 22/11: columna para guardar el distrito en las predicciones de tráfico.
# Las filas que ya existen (NO2 y PM10) quedan con id_distrito = NULL.
spark.sql("ALTER TABLE gold.fact_prediccion ADD COLUMNS (id_distrito INT)")

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


DataFrame[]

In [4]:
# fact_ruido
spark.sql(f"DROP TABLE IF EXISTS {tabla('fact_ruido')}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabla("fact_ruido")}(
        id_tiempo INT NOT NULL,
        id_sonometro STRING NOT NULL,
        id_distrito INT NOT NULL,
        nivel_ruido_db DOUBLE
    )
    USING {FORMATO}
    {ubicacion("fact_ruido")}
    """)

26/11/24 20:53:18 WARN HadoopFSUtils: The directory gs://tfm_unir1/gold/fact_ruido was not found. Was it deleted very recently?


DataFrame[]

## 5. Comprobacion

In [5]:
spark.sql(f"SHOW TABLES IN {ESQUEMA}").show(50)

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|     gold|            aire|      false|
|     gold|dim_contaminante|      false|
|     gold|    dim_distrito|      false|
|     gold|      dim_espira|      false|
|     gold|    dim_estacion|      false|
|     gold|   dim_sonometro|      false|
|     gold|      dim_tiempo|      false|
|     gold|       fact_aire|      false|
|     gold|      fact_meteo|      false|
|     gold|  fact_poblacion|      false|
|     gold| fact_prediccion|      false|
|     gold|      fact_ruido|      false|
|     gold|    fact_trafico|      false|
|     gold| fact_zona_verde|      false|
+---------+----------------+-----------+

